In [1]:
import torch 
import torch.nn.functional as F
from torch.nn.utils import clip_grad_norm_
from torch.utils.tensorboard import SummaryWriter
import torch_geometric 
from torch_geometric.loader import DataLoader as gDataLoader
import data 
from data import ProcessData
import model.base
from model.base import Transformer
import model.multiheads 
from model.multiheads import Transformer as TransformerMultiHead
import utils 
from utils import monotonic_annealer, get_mask, parallel_f, seed_torch
import rdkit 
from rdkit.Chem import MolFromSmiles as get_mol 
import datetime 
import os
from tqdm import tqdm
rdkit.rdBase.DisableLog('rdApp.*') # Disable rdkit warnings
seed_torch()
writer = SummaryWriter()
device = torch.device("cuda" if torch.cuda.is_available else "cpu")
cur_time = datetime.datetime.now()

DATA_PATH = 'data/chembl24_canon_train.pickle'
MAX_LEN = 30 
BATCH = 128 

D_MODEL = 512 
D_LATENT = 256
D_FF = 1024
N_HEADS = 8 
N_LAYERS = 6 
DROPOUT = 0.5 

LR = 5e-4
N_EPOCHS = 40 
KL_START = 0
KL_W_START = 0.001
KL_W_END = 0.005

SAVE_NAME = 'your save name'

In [ ]:
Data = ProcessData(DATA_PATH, MAX_LEN)
data_list, smi_list, vocab, inv_vocab, gvocab, max_len = (Data.process(),
                                                          Data.smi_list,
                                                          Data.vocab,
                                                          Data.inv_vocab,
                                                          Data.gvocab,
                                                          Data.max_len)
train_loader = gDataLoader(data_list, batch_size=BATCH, shuffle=True)  
print(f'\nNumber of data: {len(data_list)}')
print(f'\nSMILES Vocab:\n\t {vocab}')
print(f'\nGraph Vocab:\n\t {gvocab}')

In [ ]:
model = TransformerMultiHead(
    d_model=D_MODEL,
    d_latent=D_LATENT,
    d_ff=D_FF,
    num_head=N_HEADS,
    num_layer=N_LAYERS,
    dropout=DROPOUT,
    vocab=vocab,
    gvocab=gvocab
).to(device)

optim = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=1e-6)

def loss_fn(pred, tgt, mu, sigma, beta) :
    reconstruction_loss = F.nll_loss(pred.reshape(-1, len(vocab)), tgt.reshape(-1), ignore_index=vocab['<PAD>'])
    kl_loss = -0.5 * torch.sum(1 + sigma - mu.pow(2) - sigma.exp()).mean() / BATCH
    return  reconstruction_loss + kl_loss * beta, reconstruction_loss, kl_loss

annealer = monotonic_annealer(N_EPOCHS, KL_START, KL_W_START, KL_W_END)

In [ ]:
def read_gen_smi(t) : 
    smiles = ''.join([inv_vocab[i] for i in t])
    smiles = smiles.replace("<START>", "").replace("<PAD>", "").replace("<END>","")
    return smiles 
def get_valid(smi) : 
    return smi if get_mol(smi) else None 
def get_novel(smi) : 
    return smi if smi not in smi_list else None 

In [10]:
for epoch in range(N_EPOCHS) :
    train_loss, val_loss, recon_loss, kl_loss = 0, 0, 0, 0
    beta = annealer[epoch]

    model.train()
    # for src in train_loader :
    for src in tqdm(train_loader, desc=f'Epoch {epoch + 1}/{N_EPOCHS} - Train') : 
        src = src.to(device)
        tgt = src.clone().smi.to(device)
        tgt_mask = get_mask(tgt[:, :-1], vocab) 
        pred, mu, sigma = model(src, tgt[:, :-1], None, tgt_mask)
        loss, recon, kl = loss_fn(pred, tgt[:, 1:], mu, sigma, beta)

        train_loss += loss.item()
        recon_loss += recon.item()
        kl_loss += kl.item()

        loss.backward(), optim.step(), optim.zero_grad(), clip_grad_norm_(model.parameters(), 0.5)

    model.eval()
    gen_mol = torch.empty(0).to(device)
    with torch.no_grad() : 
        # for _ in range(60) :
        for _ in tqdm(range(60), desc='Generating Molecules...') : 
            z = torch.randn(500, D_LATENT).to(device)
            tgt = torch.zeros(500, 1, dtype=torch.long).to(device)

            for _ in range(max_len - 1) : 
                pred = model.inference(z, tgt, None, get_mask(tgt, vocab).to(device))
                _, idx = torch.topk(pred, 1, dim=-1)
                idx = idx[:, -1, :]
                tgt = torch.cat([tgt, idx], dim=1)

            gen_mol = torch.cat([gen_mol, tgt], dim=0)


        gen_mol = gen_mol.tolist() 
        gen_mol = parallel_f(read_gen_smi, gen_mol)
        valid_mol = parallel_f(get_valid, gen_mol)
        valid_mol = [m for m in valid_mol if m != None]
        unique_mol = set(valid_mol)
        uniqueness = (len(unique_mol) / len(valid_mol)) * 100 if valid_mol else 0
        novel_mol = [m for m in parallel_f(get_novel, unique_mol) if m is not None]
        novelty = (len(novel_mol) / len(unique_mol)) * 100 if unique_mol else 0
        validity = (len(valid_mol) / 30000) * 100 


        with open(f'genmol-train/{cur_time}.txt', 'a') as file : 
            if epoch == 0 : 
                file.write('Model Parameters:\n')
                for name, value in arg.__dict__.items() : 
                    file.write(f'\t{name} : {value}\n')
            file.write(f"Epoch: {epoch + 1} --- Train Loss: {train_loss / len(train_loader):3f}\n")
            file.write(f'Validity: {validity:.2f}% --- Uniqueness: {uniqueness:.2f}% --- Novelty: {novelty:.2f}%')

            for i, m in enumerate(set(novel_mol)) : 
                file.write(f'{i+1}. {m}\n')

    writer.add_scalar('Loss/Train', train_loss / len(train_loader), epoch)
    writer.add_scalar('Loss/Reconstruction', recon_loss / len(train_loader), epoch)
    writer.add_scalar('Loss/KL', kl_loss / len(train_loader), epoch)
    writer.add_scalar('Metric/Uniqueness', uniqueness, epoch)
    writer.add_scalar('Metric/Validity', validity, epoch)

    for i, m in enumerate(set(novel_mol)) : 
        print(f'{i+1}. {m}')
    print(f'Epoch: {epoch + 1}:')
    print(f'\tTrain Loss: {train_loss / len(train_loader):.3f} --- Reconstruction Loss: {recon_loss / len(train_loader):.3f} --- KL Loss: {kl_loss / len(train_loader):.3f} --- Beta: {beta:5f}')
    print(f'\tValidity: {validity:.2f}% --- Uniqueness: {uniqueness:.2f}% --- Novelty: {novelty:.2f}%\n')

  0%|                                                                                                                                                                                              | 0/703 [00:00<?, ?it/s]

100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 703/703 [00:46<00:00, 15.18it/s]


1. CCCCCCCCCCCCCCCCCCCCCCCCCCCCCC

Epoch : 0 | Loss : 1158.4624224538316
Uniqueness : 0.0033333333333333335 | Novelty : 100.0 | Validity : 100.0


100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 703/703 [00:45<00:00, 15.31it/s]


1. CCCCCCCCCCCCCCCCCCCCCCCCCCCCCC

Epoch : 1 | Loss : 1.385938265415208
Uniqueness : 0.0033334444481482716 | Novelty : 100.0 | Validity : 99.99666666666667


...

1. COCCCOCCCNCCCCNCCCCOCCOCCCOCCO

2. COCCCNC(=O)CNCCCNC(=O)CCNCCCCO

3. COc1ccccccc1CNCCNC(=O)CCOCCOCC

4. CCCCCCCCCCCCCCCCCCCCCCCCCC(=O)

5. COc1cccc(C=Cc2ccccccc2OC)ccc1O

6. O=C(O)CCCCCCCCCCCCCCCCCCCCCCCC

7. COCCCCNCCCCNCCCCCCOCCCOCCCCOCC

8. COc1ccccccc1C(=O)NCCOCCCCCCCCO

9. COCCOCCCNC(=O)CCNCCCCCNCCCOCCC

10. COc1ccccccc1OCCNCCCOc1ccccccc1

11. COCCCNC(=O)CCNCCCCNCCCCCCOCCCO

12. COCCCNCCCCNCCCCCOCCCCOCCCCCOCC

13. COCCCCCCCCCCCCNCCCCCCCCCCCCOCC

14. COc1ccccccc1CNCCCNCCCCOCCCCCCC

15. COCCCNC(=O)CCNCCCCNC(=O)CCCCOC

16. COc1ccccccc1CNC(=O)NCCCCNCCCOC

17. COc1ccccccc1CNCCNCCCOCCCCCCCCO

18. COCCCNCCCNCCCCCNCCCCOCCCOCCCOC

19. COc1ccccccc1CNCCCOCCCCCCCCCCCC

20. CCOC(=O)C(C)NC(=O)NC(C)C(C)CC

21. COCCCNCCCCCNCCCCCCCCOCCCCOCCCC

22. COc1ccccccc1OCCCNCCCOCCCCOCCCC

23. COc1ccccccc1CNCCNC(=O)CCOCCCCO

24. CCCCCCCCCCCCCCCCCCCCCCCCNCCCCN

25. CCCCCCCCCCCCCCCCCCCCCCCCCCCCNC

26. COc1ccccccc1CNC(=O)NCCCNCCSCCC

27. COc1ccccccc1CNCCNC(=O)CCOCCCCC

28. COc1ccccccc1CNCCNC(=O)CSCCCCCO

29

100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 703/703 [00:46<00:00, 15.23it/s]


1. COCCCCCCCCCCCCCCCCNCCCOCCOCCOC

2. COc1ccccccc1OCCCNCCCCCCCCCCCCO

3. COc1ccccccc1CNCCNCCNCCCNCCCOC

4. CCCCCCCCCCCCCCCCCCCCCCCCCC(=O)

5. COc1ccccccc1CNCCNC(=O)NCCCSCCC

6. COCCOCCCNCCCCCOCCCOCCCCOCCCCOC

7. COCCNCCCNCCCCNCCCCOCCOCCCOCCCO

8. COCCNCCCNCCCCNCCCCOc1ccccccc1C

9. COC(=O)CN1CCN(C(=O)CCOC)C1

10. COCCCCCCCCCCCCCCCCCCCCCNCCNCCC

11. COc1ccccccc1OCCNCCNCCCCCCOCCCC

12. COCCNC(=O)CNCCCNCCCCNCCCOC

13. O=C(O)CCCCCCCCCCCCCCCCCCCCCCCC

14. COc1cccc(C(=O)NCCCCCCC(=O)O)c1

15. COc1ccccccc1CNC(=O)NCCSCCCCNCC

16. COCCNCCCNCCCNCCCOCCOCOCCOCOCCO

17. COc1ccccccc1CNC(=O)NCCSCCCCCOC

18. COCCNCCCNCCCCOCCOCCOCCOCCCO

19. COc1ccc(OC)c(NC(=O)CCCCC)c1F

20. COc1ccccccc1CNCCNCCCSCCCCOCCCC

21. COCCNC(=O)CNCCCCNCCCCCOCCCOC

22. COC(=O)CNCCCNC(=O)C(=O)C(=O)C

23. COCCNCCCNCCCOc1ccccccc1SCCCOCC

24. COc1ccccccc1CNC(=O)NCCNCCCCOCC

25. COc1cccc(CNC(=O)NCCCCCOC)c1F

26. COCCNCCCNCCCCNCCCOc1ccccccc1OC

27. O=C(NCCCCCCCCCCCCCCCCCCCCO)CCC

28. COc1ccccccc1CNC(=O)NCCCCNCCCOC

29. COc1ccccccc1NCCN

100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 703/703 [00:46<00:00, 15.21it/s]


1. COCCNCCCNCCCCNCCCCOCCCOCCCCO

2. COCCCCNCCCCCNCCCCCCCCOCCCCCOCC

3. CCN(C)CCCNC(=O)c1cc(Cl)c(Cl)cc1

4. COc1ccc(CNC(=O)Nc2ccccc2F)cc1

5. COCCOCCOCCOCCCOCCCOCCCOCCCOCCC

6. COc1ccc(CNC(=O)CN2CCCCCCC2)cc1

7. CCCCCCCCCCCCCCCCCCCCCCCCCC(=O)

8. COc1ccc(CNC(=O)NCCCNC(C)C)cc1O

9. O=C(O)C(CO)CNC(=O)c1ccc(Cl)cc1

10. COCCCNC(=O)CNCCCCCCCCCCCCCCCCO

11. COCCCNCCCNCCCCCNCCCCCCCOCCCCCO

12. COCCOCCNCCCOCCCOCCOCCCOCCCO

13. CCOCCNCCCNCCCNCCCCOCCCCO

14. COCCCNC(=O)CNCCCCNCCCCCOCCCOC

15. COCCCCCCCCCCCCCCCCCCCCCCCCCCCN

16. COCCNCCCNCCCNCCCCNCCCCOCCCCO

17. COCCCNC(=O)NCCCNC(=O)NCCCCCCCO

18. COCCNCCCNCCCNCCCCNCCCCCOCCCOCC

19. COCCNCCCNCCCCNCCCOCCOCCCOCCCO

20. COCCNC(=O)NCCCNCCCCCCCCCCCOCCC

21. CCOCCNC(=O)CNCCCCNCCCCCCCCCCCC

22. O=C(O)CCCCCCCCCCCCCCCCCCCCCCCC

23. COCCNCCCNCCCNCCCCCNCCCCCOCCCOC

24. COCCCNC(=O)NCCCNC(=O)CCCCCCCOC

25. COCCCNC(=O)C(C)NC(=O)CCCCCCCCN

26. COCCNCCCNCCCCNCCCCCCCOCCCCCO

27. COCCCNCCCNCCCNCCCCOCCOCCOCCCOC

28. COCCCNCCCCOCCCCOCCCCCNCCCCOCCC

29. COc1ccc(C(=O)N

In [11]:
for i in train_loader : 
    print(i) 
    break

MyDataBatch(x=[2047], edge_index=[2, 4334], smi=[128, 31], batch=[2047], ptr=[129])
